In [1]:
import pandas as pd
import numpy as np
import pyarrow
import json
import os
import time

## Dataset A: User Input
This represents User Input Data. In production, this is the most volatile source because humans are unpredictable. You can expect issues like SQL injection attempts, inconsistent casing, typos in email addresses, and "impossible" values (like an age of 500). Before using this in an ML pipeline, I would apply strict schema validation, regex-based email verification, and outlier detection to ensure the numerical distributions aren't skewed by nonsense entries.

In [2]:
dataset_a = [
    {"username": "jdoe", "email": "john@example.com", "age": 25, "signup_date": "2026-01-15"},
    {"username": "alice_w", "email": "alice@site.org", "age": 30, "signup_date": "2026-01-16"},
    {"username": "bob_build", "email": "bob_at_email.com", "age": 42, "signup_date": "2026-01-17"}, # Invalid Email
    {"username": "charlie", "email": "char@web.com", "age": "twenty", "signup_date": "2026-01-18"}, # Age as string
    {"username": "", "email": "ghost@void.com", "age": 19, "signup_date": "2026-01-19"},            # Empty username
    {"username": "tech_guru", "email": "guru@code.io", "age": 55, "signup_date": "2026-01-20"},
    {"username": "data_viz", "email": "viz@chart.com", "age": 28, "signup_date": "2026-01-21"},
    {"username": "last_one", "email": "last@test.com", "age": 33, "signup_date": "2026-01-22"}
]

## Dataset B: Application Logs
This is System-Generated Data. These sources are usually structured but high-volume. The main quality issues include "log drift" (where a code update changes the message format unexpectedly) or missing entries due to network latency in log shipping. For an ML pipeline (like anomaly detection), I would validate the timestamp formats and ensure the level field matches a predefined set of categories (INFO/WARN/ERROR) to avoid categorical explosion.

In [3]:
dataset_b = [
    {"timestamp": "2026-03-02 10:00:01", "service": "AuthService", "level": "INFO", "message": "User login successful"},
    {"timestamp": "2026-03-02 10:05:22", "service": "PaymentGateway", "level": "ERROR", "message": "Connection timeout"},
    {"timestamp": "2026-03-02 10:10:00", "service": "Inventory", "level": "INFO", "message": "Stock updated for SKU-102"},
    {"timestamp": "2026-03-02 10:12:45", "service": "AuthService", "level": "WARNING", "message": "Failed login attempt"},
    {"timestamp": "2026-03-02 10:15:30", "service": "PaymentGateway", "level": "INFO", "message": "Retry successful"},
    {"timestamp": "2026-03-02 10:20:11", "service": "Inventory", "level": "WARNING", "message": "Low stock alert"},
    {"timestamp": "2026-03-02 10:25:00", "service": "AuthService", "level": "INFO", "message": "Password changed"},
    {"timestamp": "2026-03-02 10:30:15", "service": "PaymentGateway", "level": "ERROR", "message": "Invalid API key"},
    {"timestamp": "2026-03-02 10:35:40", "service": "Inventory", "level": "INFO", "message": "New item added"},
    {"timestamp": "2026-03-02 10:40:05", "service": "AuthService", "level": "INFO", "message": "Session expired"}
]

## Dataset C: Product Inventory
This represents an Internal Database source. It is generally the most reliable as it has already passed through transactional constraints (PostgreSQL/MySQL types). However, issues like "stale data" (the last_updated field being too old) or synchronization mismatches between different warehouses are common. Validation should focus on referential integrity and ensuring quantity_in_stock is never a negative value.

In [4]:
dataset_c = [
    {"sku": "SKU-001", "product_name": "Ergonomic Chair", "quantity_in_stock": 150, "warehouse_location": "East-1", "last_updated": "2026-03-01"},
    {"sku": "SKU-002", "product_name": "Mechanical Keyboard", "quantity_in_stock": 45, "warehouse_location": "West-2", "last_updated": "2026-02-28"},
    {"sku": "SKU-003", "product_name": "UltraWide Monitor", "quantity_in_stock": 12, "warehouse_location": "East-1", "last_updated": "2026-03-02"},
    {"sku": "SKU-004", "product_name": "USB-C Dock", "quantity_in_stock": 200, "warehouse_location": "Central-3", "last_updated": "2026-03-01"},
    {"sku": "SKU-005", "product_name": "Noise Cancelling Headphones", "quantity_in_stock": 85, "warehouse_location": "West-2", "last_updated": "2026-02-25"},
    {"sku": "SKU-006", "product_name": "Standing Desk", "quantity_in_stock": 5, "warehouse_location": "East-1", "last_updated": "2026-03-02"}
]

In [5]:
def validate_registrations(entries):
    valid_entries = []
    invalid_entries = []

    for entry in entries:
        username = entry.get('username', '')
        email = entry.get('email', '')
        age = entry.get('age')

        is_username_valid = len(str(username).strip()) > 0
        is_email_valid = "@" in str(email)

        try:
            age_int = int(age)
            is_age_valid = age_int > 0
        except:
            is_age_valid = False

        if is_username_valid and is_email_valid and is_age_valid:
            valid_entries.append(entry)
        else:
            invalid_entries.append(entry)

    return valid_entries, invalid_entries

valid, invalid = validate_registrations(dataset_a)

print(f"Validation Results:")
print(f"Total Entries:   {len(dataset_a)}")
print(f"Valid Entries:   {len(valid)}")
print(f"Invalid Entries: {len(invalid)}")

Validation Results:
Total Entries:   8
Valid Entries:   5
Invalid Entries: 3


In [6]:
np.random.seed(16)

rows = 500
rides = pd.DataFrame({
    'ride_id' : range(1, rows + 1),
    'driver_id' : np.random.randint(1000, 1050, size=rows),
    'distance_km' : np.random.uniform(1.0, 30.0, size=rows),
    'fare_usd' : np.random.uniform(5.0, 75.0, size=rows),
    'ride_type' : np.random.choice(['standard', 'premium', 'pool'], size=rows),
    'city' : np.random.choice(["Berlin", "Seoul", "Nairobi", "Toronto", "Lima"], size=rows),
    'timestamp' : pd.date_range(start='2025-01-01', periods=rows, freq='min')
})

In [7]:
rides.to_csv('rides.csv', index=False)
rides.to_json('rides.json', orient='records', lines=True)
rides.to_parquet('rides.parquet', index=False)

C:\Users\TEXNO\AppData\Local\Temp\ipykernel_10528\2960519296.py:2: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  rides.to_json('rides.json', orient='records', lines=True)


In [8]:
files = ["rides.csv", "rides.json", "rides.parquet"]
sizes_kb = {f: os.path.getsize(f) / 1024 for f in files}

for f, size in sizes_kb.items():
    print(f"{f}: {size:.2f} KB")

rides.csv: 39.52 KB
rides.json: 72.42 KB
rides.parquet: 20.34 KB


In [9]:
ratio = sizes_kb["rides.csv"] / sizes_kb["rides.parquet"]
print(f"\nParquet is {ratio:.2f}x smaller than CSV.")


Parquet is 1.94x smaller than CSV.


In [10]:
df_csv = pd.read_csv('rides.csv')
df_json = pd.read_json('rides.json', orient='records', lines=True)
df_parquet = pd.read_parquet('rides.parquet')

In [11]:
print(f"Shapes match: {rides.shape == df_csv.shape == df_json.shape == df_parquet.shape}")

Shapes match: True


In [12]:
print(f"\nOriginal Type: {rides['timestamp'].dtype}")
print(f"CSV Type:      {df_csv['timestamp'].dtype} (Lost datetime type, became string)")
print(f"JSON Type:     {df_json['timestamp'].dtype} (Preserved datetime via pandas magic)")
print(f"Parquet Type:  {df_parquet['timestamp'].dtype} (Perfectly preserved)")


Original Type: datetime64[us]
CSV Type:      str (Lost datetime type, became string)
JSON Type:     datetime64[ms] (Preserved datetime via pandas magic)
Parquet Type:  datetime64[us] (Perfectly preserved)


1. Which format preserved types most faithfully?
Parquet is the clear winner. Unlike CSV (which treats everything as text) or JSON (which often requires manual conversion of dates), Parquet stores the "metadata" or schema directly within the file. When you read a Parquet file back into Python, an integer stays an integer and a timestamp stays a timestamp without any extra code.

2. Which format was the smallest?
Parquet was significantly smaller. This is because Parquet uses columnar storage and sophisticated compression algorithms (like Snappy or Gzip). It only stores unique values and repetitions efficiently, whereas CSV and JSON repeat every character and delimiter for every single row.

3. When would you choose each format?
Choose CSV when you need portability. If you need to email a dataset to a colleague who only uses Excel or a text editor, CSV is the universal standard that "just works" everywhere.

Choose JSON (Lines) for web logging and streaming. Because each line is a valid independent object, it is perfect for real-time application logs. If the file transfer is interrupted, you only lose the last line, not the entire file structure.

Choose Parquet for Machine Learning and Big Data. It is the industry standard for data lakes (like AWS S3). It allows you to load only the columns you need into memory, saving massive amounts of RAM and processing time during model training.

In [13]:
num_rows = 200_000
num_cols = 20
col_names = [f'feature_{i}' for i in range(num_cols)]

df = pd.DataFrame(
    np.random.randn(num_rows, num_cols),
    columns=col_names
)

In [14]:
%time df['feature_0'].mean()

CPU times: total: 0 ns
Wall time: 2.09 ms


np.float64(0.0015185741060130698)

In [15]:
def manual_row_sum(df, limit=10000):
    total = 0
    for i in range(limit):
        total += df.iloc[i]['feature_0']
    return total

%time manual_row_sum(df)

CPU times: total: 188 ms
Wall time: 192 ms


np.float64(69.522397535172)

In [16]:
arr = df.to_numpy()

In [17]:
%time arr[:, 0].mean()

CPU times: total: 0 ns
Wall time: 446 μs


np.float64(0.0015185741060130698)

In [18]:
def numpy_row_sum(arr, limit=10000):
    total = 0
    for i in range(limit):
        total += arr[i, 0]
    return total

%time numpy_row_sum(arr)

CPU times: total: 15.6 ms
Wall time: 3.01 ms


np.float64(69.522397535172)

1. Why is column access so much faster?
Pandas uses Columnar Storage. All values in a column are stored together in one contiguous block of memory. Accessing a column is a single "grab," while row iteration forces Pandas to create a new Series object for every row, which is incredibly slow.

2. Why does the gap narrow with NumPy?
NumPy lacks the "overhead" of Pandas. It doesn't create complex objects (like a Series with an Index) during iteration; it just accesses raw numbers. Since it's written in C, it handles loops much faster, though vectorized column operations still win.

3. What does this tell us about memory layout?
It proves that data is laid out in contiguous blocks.

Pandas is optimized for columns; it wants to process entire features at once.

CPU Efficiency: Moving through memory sequentially (vectorization) allows the CPU to cache data effectively. Jumping around (manual loops) causes "cache misses," which kills performance.

In [19]:
np.random.seed(42)
rows = 100_000

df = pd.DataFrame({
    'f1': np.random.randn(rows),
    'f2': np.random.randn(rows),
    'f3': np.random.randn(rows),
    'f4': np.random.randn(rows),
    'f5': np.random.randn(rows),
    'category': np.random.choice(["alpha", "beta", "gamma"], size=rows)
})

In [20]:
df.to_csv('data.csv', index=False)
df.to_parquet('data.parquet', index=False)

csv_size = os.path.getsize("data.csv")
parquet_size = os.path.getsize("data.parquet")

csv_size, parquet_size

(10482182, 5044687)

In [21]:
df_csv_loaded = pd.read_csv("data.csv")
df_parquet_loaded = pd.read_parquet("data.parquet")

csv_size = os.path.getsize("data.csv")
parquet_size = os.path.getsize("data.parquet")

csv_size, parquet_size

(10482182, 5044687)

In [22]:
print(df_csv_loaded.dtypes)

f1          float64
f2          float64
f3          float64
f4          float64
f5          float64
category        str
dtype: object


In [23]:
print(df_parquet_loaded.dtypes)

f1          float64
f2          float64
f3          float64
f4          float64
f5          float64
category        str
dtype: object


### Data Type Preservation: CSV vs Parquet

In this experiment, the data types appeared the same when reading both the CSV and the Parquet files:

- All numeric columns were read as float64.
- The category column was read as string.

This happened because the dataset only contained simple float values and plain strings. Pandas was able to correctly infer the numeric columns when reading the CSV file, so the resulting dtypes matched those stored in the Parquet file.

However, this does not mean CSV always preserves types. CSV is a text-based format and does not store schema information. Pandas must infer types when reading it, which can lead to differences in more complex datasets (e.g., datetime columns, categorical types, nullable integers, booleans).

Parquet, on the other hand, is a binary format that stores schema metadata. It preserves the exact data types that were written, even for more complex types.

In this specific case, both formats produced the same dtypes because the dataset only contained simple floats and strings.

In [24]:
%time df_csv = pd.read_csv("data.csv")

CPU times: total: 250 ms
Wall time: 254 ms


In [25]:
%time df_parquet = pd.read_parquet("data.parquet")

CPU times: total: 62.5 ms
Wall time: 19.3 ms


In [26]:
%time df_category = pd.read_parquet('data.parquet', columns=['category'])

CPU times: total: 0 ns
Wall time: 8.45 ms


In [27]:
%time df_full = pd.read_parquet("data.parquet")

CPU times: total: 0 ns
Wall time: 11.6 ms


### Parquet vs CSV

Using `%time`, we observed:

- **Full Parquet read:** Wall time ~17.2 ms
- **Category-only Parquet read:** Wall time ~13.3 ms

Reading only the `category` column is faster because **Parquet is a columnar storage format**. Each column is stored separately on disk, so pandas can load just the requested column without reading the rest of the file.

CSV, by contrast, is a **row-based text format**. Each row contains all column values in sequence. Even if we want only a single column, pandas must read and parse the entire file row by row, which makes selective reads inefficient.

This demonstrates one of the key advantages of Parquet for analytical workloads: **columnar storage enables fast, memory-efficient access to specific columns**.

In [28]:
flat_data = [
    # Penguin UK
    ["The Silent Forest", "Alice Green", "Paperback", "Penguin Books", "UK", 15.99],
    ["The Silent Forest", "Alice Green", "E-book", "Penguin Books", "UK", 9.99],
    ["Ocean Depths", "Alice Green", "Paperback", "Penguin Books", "UK", 14.99],
    ["Ocean Depths", "Alice Green", "E-book", "Penguin Books", "UK", 8.99],

    # HarperCollins US
    ["Data Smart", "Bob Smith", "Paperback", "HarperCollins", "USA", 29.99],
    ["Data Smart", "Bob Smith", "E-book", "HarperCollins", "USA", 19.99],
    ["AI Basics", "Bob Smith", "Paperback", "HarperCollins", "USA", 24.99],

    # Vintage Canada
    ["History of Time", "Clara Lee", "Paperback", "Vintage", "Canada", 18.99],
    ["History of Time", "Clara Lee", "E-book", "Vintage", "Canada", 11.99],
    ["Modern Physics", "Clara Lee", "Paperback", "Vintage", "Canada", 22.99],

    # O’Reilly US
    ["Python 101", "David Kim", "Paperback", "O'Reilly Media", "USA", 34.99],
    ["Python 101", "David Kim", "E-book", "O'Reilly Media", "USA", 24.99],
]

flat_df = pd.DataFrame(flat_data, columns=[
    "title", "author", "format", "publisher_name", "publisher_country", "price"
])

flat_df

,title,author,format,publisher_name,publisher_country,price
0,The Silent Forest,Alice Green,Paperback,Penguin Books,UK,15.99
1,The Silent Forest,Alice Green,E-book,Penguin Books,UK,9.99
2,Ocean Depths,Alice Green,Paperback,Penguin Books,UK,14.99
3,Ocean Depths,Alice Green,E-book,Penguin Books,UK,8.99
4,Data Smart,Bob Smith,Paperback,HarperCollins,USA,29.99
5,Data Smart,Bob Smith,E-book,HarperCollins,USA,19.99
6,AI Basics,Bob Smith,Paperback,HarperCollins,USA,24.99
7,History of Time,Clara Lee,Paperback,Vintage,Canada,18.99
8,History of Time,Clara Lee,E-book,Vintage,Canada,11.99
9,Modern Physics,Clara Lee,Paperback,Vintage,Canada,22.99


In [29]:
publishers = flat_df[["publisher_name", "publisher_country"]].drop_duplicates().reset_index(drop=True)
publishers["publisher_id"] = publishers.index + 1
publishers = publishers[["publisher_id", "publisher_name", "publisher_country"]]
publishers

,publisher_id,publisher_name,publisher_country
0,1,Penguin Books,UK
1,2,HarperCollins,USA
2,3,Vintage,Canada
3,4,O'Reilly Media,USA


In [30]:
books = flat_df.merge(publishers, on=["publisher_name", "publisher_country"])
books = books[["title", "author", "format", "publisher_id", "price"]]
books

,title,author,format,publisher_id,price
0,The Silent Forest,Alice Green,Paperback,1,15.99
1,The Silent Forest,Alice Green,E-book,1,9.99
2,Ocean Depths,Alice Green,Paperback,1,14.99
3,Ocean Depths,Alice Green,E-book,1,8.99
4,Data Smart,Bob Smith,Paperback,2,29.99
5,Data Smart,Bob Smith,E-book,2,19.99
6,AI Basics,Bob Smith,Paperback,2,24.99
7,History of Time,Clara Lee,Paperback,3,18.99
8,History of Time,Clara Lee,E-book,3,11.99
9,Modern Physics,Clara Lee,Paperback,3,22.99


In [31]:
reconstructed = books.merge(publishers, on="publisher_id")
reconstructed = reconstructed[flat_df.columns]

In [32]:
flat_df.equals(reconstructed)

True

In [33]:
document_data = []

for (title, author), group in flat_df.groupby(["title", "author"]):
    publisher_name = group.iloc[0]["publisher_name"]
    publisher_country = group.iloc[0]["publisher_country"]

    editions = group[["format", "price"]].to_dict(orient="records")

    document = {
        "title": title,
        "author": author,
        "publisher": {
            "name": publisher_name,
            "country": publisher_country
        },
        "editions": editions
    }

    document_data.append(document)

document_data

[{'title': 'AI Basics',
  'author': 'Bob Smith',
  'publisher': {'name': 'HarperCollins', 'country': 'USA'},
  'editions': [{'format': 'Paperback', 'price': 24.99}]},
 {'title': 'Data Smart',
  'author': 'Bob Smith',
  'publisher': {'name': 'HarperCollins', 'country': 'USA'},
  'editions': [{'format': 'Paperback', 'price': 29.99},
   {'format': 'E-book', 'price': 19.99}]},
 {'title': 'History of Time',
  'author': 'Clara Lee',
  'publisher': {'name': 'Vintage', 'country': 'Canada'},
  'editions': [{'format': 'Paperback', 'price': 18.99},
   {'format': 'E-book', 'price': 11.99}]},
 {'title': 'Modern Physics',
  'author': 'Clara Lee',
  'publisher': {'name': 'Vintage', 'country': 'Canada'},
  'editions': [{'format': 'Paperback', 'price': 22.99}]},
 {'title': 'Ocean Depths',
  'author': 'Alice Green',
  'publisher': {'name': 'Penguin Books', 'country': 'UK'},
  'editions': [{'format': 'Paperback', 'price': 14.99},
   {'format': 'E-book', 'price': 8.99}]},
 {'title': 'Python 101',
  'autho

In [34]:
flat_df[flat_df["author"] == "Alice Green"]

,title,author,format,publisher_name,publisher_country,price
0,The Silent Forest,Alice Green,Paperback,Penguin Books,UK,15.99
1,The Silent Forest,Alice Green,E-book,Penguin Books,UK,9.99
2,Ocean Depths,Alice Green,Paperback,Penguin Books,UK,14.99
3,Ocean Depths,Alice Green,E-book,Penguin Books,UK,8.99


In [35]:
books[books["author"] == "Alice Green"]

,title,author,format,publisher_id,price
0,The Silent Forest,Alice Green,Paperback,1,15.99
1,The Silent Forest,Alice Green,E-book,1,9.99
2,Ocean Depths,Alice Green,Paperback,1,14.99
3,Ocean Depths,Alice Green,E-book,1,8.99


In [36]:
[doc for doc in document_data if doc["author"] == "Alice Green"]

[{'title': 'Ocean Depths',
  'author': 'Alice Green',
  'publisher': {'name': 'Penguin Books', 'country': 'UK'},
  'editions': [{'format': 'Paperback', 'price': 14.99},
   {'format': 'E-book', 'price': 8.99}]},
 {'title': 'The Silent Forest',
  'author': 'Alice Green',
  'publisher': {'name': 'Penguin Books', 'country': 'UK'},
  'editions': [{'format': 'Paperback', 'price': 15.99},
   {'format': 'E-book', 'price': 9.99}]}]

In [37]:
flat_df["price"].mean()

np.float64(19.90666666666667)

In [38]:
books["price"].mean()

np.float64(19.90666666666667)

In [40]:
prices = [
    edition["price"]
    for book in document_data
    for edition in book["editions"]
]

sum(prices) / len(prices)

19.906666666666666

In [42]:
flat_df.loc[flat_df['publisher_name'] == 'Vintage', 'publisher_country'] = 'USA'

In [43]:
(flat_df['publisher_name'] == 'Vintage').sum()

np.int64(3)

In [45]:
publishers.loc[publishers['publisher_name'] == 'Vintage', 'publisher_country'] = "USA"


In [46]:
(publishers['publisher_name'] == 'Vintage').sum()

np.int64(1)

In [47]:
document_data

[{'title': 'AI Basics',
  'author': 'Bob Smith',
  'publisher': {'name': 'HarperCollins', 'country': 'USA'},
  'editions': [{'format': 'Paperback', 'price': 24.99}]},
 {'title': 'Data Smart',
  'author': 'Bob Smith',
  'publisher': {'name': 'HarperCollins', 'country': 'USA'},
  'editions': [{'format': 'Paperback', 'price': 29.99},
   {'format': 'E-book', 'price': 19.99}]},
 {'title': 'History of Time',
  'author': 'Clara Lee',
  'publisher': {'name': 'Vintage', 'country': 'Canada'},
  'editions': [{'format': 'Paperback', 'price': 18.99},
   {'format': 'E-book', 'price': 11.99}]},
 {'title': 'Modern Physics',
  'author': 'Clara Lee',
  'publisher': {'name': 'Vintage', 'country': 'Canada'},
  'editions': [{'format': 'Paperback', 'price': 22.99}]},
 {'title': 'Ocean Depths',
  'author': 'Alice Green',
  'publisher': {'name': 'Penguin Books', 'country': 'UK'},
  'editions': [{'format': 'Paperback', 'price': 14.99},
   {'format': 'E-book', 'price': 8.99}]},
 {'title': 'Python 101',
  'autho

In [48]:
count = 0
for doc in document_data:
    if doc['publisher']['name'] == 'Vintage':
        doc['publisher']['country'] = "USA"
        count += 1

In [49]:
count

2

# Representation Comparison

## 1. Finding Books by Author
- Flat: Very easy (simple filter)
- Normalized: Easy (filter books table)
- Document: Easy (list comprehension)

All three perform similarly well.


## 2. Average Price Across Editions
- Flat: Easiest (mean of column)
- Normalized: Same as flat
- Document: Slightly harder (nested loop required)

Flat and normalized are better for aggregate analysis.


## 3. Updating Publisher Country
- Flat: Must update multiple rows (data redundancy problem)
- Normalized: Update only 1 row (most efficient)
- Document: Must update multiple documents

Normalized representation is best when publisher information changes frequently.


## Final Decision

If publisher information changes frequently: Normalized

If each book is always accessed as a standalone record: Document-Oriented

If the focus is simple analytics and small datasets: Flat is acceptable but not ideal for updates.